# Amaliyot 15 — Agent va ReAct sikli

**Mos ma'ruza:** L15 — Sun'iy intellekt agentlarini yaratish  
**Capstone natijasi:** `DocumentAssistantAgent`  
**Muhit:** asosiy qism CPU'da ishlaydi; lokal Qwen bo'limi uchun Kaggle GPU va Internet kerak. API kaliti kerak emas.

Bugun tayyor agent kodini ko'chirmaymiz. Uni bosqichma-bosqich quramiz:

1. kichik NLP vositalari;
2. vosita reyestri;
3. qoidaviy rejalashtiruvchi;
4. ReAct boshqaruv sikli;
5. guardrails va capstone klassi.

**Taxminiy vaqt:** kirish 6 min · toollar 12 min · planner 12 min · ReAct 18 min · capstone 12 min · lokal Qwen 15 min · yakun 5 min.

## 0. Muhit

Asosiy lab faqat Python standart kutubxonasidan foydalanadi. Shu sababli Kaggle'da paket o'rnatish yoki internetni yoqish shart emas.

In [6]:
import json
import platform
import re
from typing import Optional

print("Python:", platform.python_version())
print("Asosiy qism tayyor. Lokal Qwen bo'limida GPU kerak bo'ladi.")

Python: 3.12.13
Asosiy qism tayyor. Lokal Qwen bo'limida GPU kerak bo'ladi.


## 1. Agent nimani boshqaradi?

Oddiy pipeline qadamlari oldindan belgilangan. Agent esa so'rov va oldingi kuzatuvlarga qarab keyingi vositani tanlaydi.

Agentning boshqaruv sikli

Bu notebookda haqiqiy LLM o'rniga tushunarli **qoidaviy planner** ishlatamiz. Shunda agent siklining mexanikasi API kalitisiz ko'rinadi. Oxirida planner qismini lokal Qwen3 modeli bilan almashtiramiz.

## 2. Agent ishlata oladigan vositalar

Tool — agent chaqira oladigan, bitta aniq vazifani bajaruvchi funksiya. Avval uchta kichik backend tayyorlaymiz: sentiment, qidiruv va qisqartirish.

Tokenizatsiya yordamchi qadamdir; u agent mavzusi emas, shuning uchun juda sodda qoldirilgan.

In [7]:
def tokenize(text: str) -> list[str]:
    """O'zbekcha apostroflarni saqlagan holda sodda tokenizatsiya."""
    return re.findall(r"[a-zA-Z0-9_ʻʼ’'-]+", text.lower())

### 2.1 Sentiment vositasi

Bu yerda maqsad kuchli klassifikator yaratish emas. Agent chaqira oladigan, kirish va chiqishi aniq funksiyani olish kifoya. Keyinchalik shu funksiyaning o'rniga oldingi capstone klassifikatorini ulash mumkin.

In [8]:
def sentiment_classify(text: str) -> dict:
    """Kichik demo sentiment vositasi."""
    tokens = set(tokenize(text))
    positive_words = {"yaxshi", "ajoyib", "zo'r", "foydali", "qulay"}
    negative_words = {"yomon", "qiyin", "sifatsiz", "muammo", "sekin"}

    positive_count = len(tokens & positive_words)
    negative_count = len(tokens & negative_words)
    if positive_count > negative_count:
        return {"label": "ijobiy", "confidence": 0.82}
    if negative_count > positive_count:
        return {"label": "salbiy", "confidence": 0.79}
    return {"label": "neytral", "confidence": 0.55}

### 2.2 Kichik bilimlar bazasi

Alohida dataset yuklamaymiz. Beshta yozuv agent va qidiruv oqimini ko'rish uchun yetarli.

In [9]:
KNOWLEDGE_BASE = [
    {"title": "NLP", "text": "NLP kompyuterga inson tilidagi matnni tahlil qilishga yordam beradi."},
    {"title": "Transformer", "text": "Transformer attention yordamida tokenlar orasidagi bog'lanishni modellashtiradi."},
    {"title": "RAG", "text": "RAG avval hujjat topadi, keyin savol va kontekstni generatorga yuboradi."},
    {"title": "Agent", "text": "Agent maqsadga qarab vosita tanlaydi, natijani kuzatadi va keyingi qadamni belgilaydi."},
    {"title": "O'zbek NLP", "text": "O'zbek tili agglutinativ bo'lgani uchun qo'shimchalar matn tahlilida muhim."},
]

In [10]:
def retrieve_docs(query: str) -> dict:
    """So'rovga eng ko'p umumiy tokeni bor hujjatni topadi."""
    query_tokens = set(tokenize(query))
    scored_documents = []
    for document in KNOWLEDGE_BASE:
        document_tokens = set(tokenize(document["text"] + " " + document["title"]))
        score = len(query_tokens & document_tokens)
        scored_documents.append((score, document))

    best_score, best_document = max(scored_documents, key=lambda item: item[0])
    return {
        "title": best_document["title"],
        "text": best_document["text"],
        "score": best_score,
    }

`retrieve_docs()` so'rov va hujjat orasidagi umumiy tokenlarni sanaydi. Bu RAG modelining o'rnini bosmaydi; u faqat agent uchun tez va deterministik qidiruv toolidir.

### 2.3 Qisqartirish vositasi

Bu ekstraktiv demo birinchi ikki gapni oladi. Agent nuqtayi nazaridan muhim narsa: funksiya matn qabul qiladi va strukturali natija qaytaradi.

In [11]:
def summarize(text: str) -> dict:
    """Birinchi ikki gapni qaytaruvchi ekstraktiv demo vosita."""
    sentences = [part.strip() for part in re.split(r"[.!?]+", text) if part.strip()]
    selected = sentences[:2]
    summary = ". ".join(selected) + ("." if selected else "")
    return {"summary": summary, "sentence_count": len(selected)}

Uchala backendni alohida sinab ko'ring. Agentni yozishdan oldin toolning o'zi ishlashiga ishonch hosil qilish debuggingni osonlashtiradi.

In [12]:
print(sentiment_classify("Bu xizmat juda yaxshi va qulay."))
print(retrieve_docs("RAG nima?"))
print(summarize("Agent vosita tanlaydi. Natijani kuzatadi. So'ng javob beradi."))

{'label': 'ijobiy', 'confidence': 0.82}
{'title': 'RAG', 'text': 'RAG avval hujjat topadi, keyin savol va kontekstni generatorga yuboradi.', 'score': 1}
{'summary': 'Agent vosita tanlaydi. Natijani kuzatadi.', 'sentence_count': 2}


## 3. Tool reyestri

Agentga faqat Python funksiyasini berish yetmaydi. Vositaning nomi, vazifasi va kutilgan kirishi ham tushunarli bo'lishi kerak.

Tool tuzilishi

Bizning qoidaviy plannerimiz `keywords` maydonidan foydalanadi. Haqiqiy LLM esa odatda `description` va argument sxemasidan ma'no chiqaradi.

In [13]:
TOOLS = {
    "sentiment_classify": {
        "function": sentiment_classify,
        "description": "Matn hissiyotini aniqlaydi.",
        "keywords": ["hissiyot", "ijobiy", "salbiy", "yaxshi", "yomon", "baho"],
    },
    "retrieve_docs": {
        "function": retrieve_docs,
        "description": "Bilimlar bazasidan mos hujjatni topadi.",
        "keywords": ["top", "qidir", "ma'lumot", "hujjat", "nima", "haqida"],
    },
    "summarize": {
        "function": summarize,
        "description": "Berilgan matnni qisqartiradi.",
        "keywords": ["xulosa", "qisqa", "qisqartir", "umumlashtir"],
    },
}

Reyestrni ko'zdan kechiring. Har bir tool bir xil shartnomaga ega: `function`, `description`, `keywords`.

In [14]:
for tool_name, tool_info in TOOLS.items():
    print(f"{tool_name:20} -> {tool_info['description']}")

sentiment_classify   -> Matn hissiyotini aniqlaydi.
retrieve_docs        -> Bilimlar bazasidan mos hujjatni topadi.
summarize            -> Berilgan matnni qisqartiradi.


## 4. Mashq 1 — keyingi vositani tanlash

Ma'ruzadagi formula:

$$a_t = \operatorname{LLM}(q, h_{t-1})$$

Biz LLM o'rniga `select_tool()` qoidaviy plannerini yozamiz. U so'rovdagi kalit so'zlarni sanaydi va tarixda ishlatilgan toolni qayta tanlamaydi.

In [15]:
def select_tool(query: str, history: list[dict], tools: dict) -> Optional[str]:
    """LLM o'rnidagi tushunarli, qoidaviy rejalashtiruvchi."""
    q = query.lower()
    used = {record["tool"] for record in history}
    best_name, best_score = None, 0
    for name, info in tools.items():
        if name in used:
            continue
        score = sum(1 for kw in info["keywords"] if kw in q)
        if score > best_score:
            best_name, best_score = name, score
    if best_score == 0:
        return None
    return best_name

Tekshiruvlar uch holatni qamrab oladi: sentiment tanlash, qidiruv tanlash va tarixdagi toolni takrorlamaslik.

In [16]:
assert select_tool("Bu mahsulot yaxshi", [], TOOLS) == "sentiment_classify"
assert select_tool("RAG haqida ma'lumot top", [], TOOLS) == "retrieve_docs"

used_history = [{"tool": "sentiment_classify"}]
assert select_tool("Bu mahsulot yaxshi", used_history, TOOLS) is None
print("Mashq 1: PASS")

Mashq 1: PASS


## 5. Mashq 2 — harakatni bajarish

Planner faqat tool nomini qaytaradi. Executor shu nomni tekshiradi va mos Python funksiyasini chaqiradi:

$$o_t = \operatorname{tool}(a_t)$$

Noma'lum nomni jimgina o'tkazib yubormang; tushunarli xato chiqaring.

In [17]:
def execute_action(tool_name: str, query: str, tools: dict) -> dict:
    """Tanlangan vositani xavfsiz tekshiradi va bajaradi."""
    if tool_name not in tools:
        raise ValueError(f"Noma'lum tool: {tool_name}")
    function = tools[tool_name]["function"]
    return function(query)

Quyidagi tekshiruv haqiqiy natijani va noto'g'ri tool uchun guardrailni sinaydi.

In [18]:
observation = execute_action("sentiment_classify", "Xizmat yaxshi", TOOLS)
assert observation["label"] == "ijobiy"

try:
    execute_action("delete_everything", "test", TOOLS)
except ValueError as error:
    print("Guardrail ishladi:", error)

Guardrail ishladi: Noma'lum tool: delete_everything


## 6. Mashq 3 — ReAct sikli

Har qadamda planner tool tanlaydi, executor uni bajaradi va natija tarixga yoziladi:

$$a_t = \operatorname{planner}(q,h_{t-1}), \qquad o_t = \operatorname{tool}(a_t)$$
$$h_t = h_{t-1} \;\Vert\; [(a_t,o_t)]$$

ReAct tarixining o'sishi

In [19]:
def format_final_answer(history: list[dict]) -> str:
    """Kuzatuvlarni foydalanuvchiga ko'rinadigan qisqa javobga aylantiradi."""
    if not history:
        return "Mos vosita topilmadi. So'rovni aniqroq yozing."
    parts = []
    for record in history:
        observation = record["observation"]
        parts.append(f"{record['tool']}: {json.dumps(observation, ensure_ascii=False)}")
    return " | ".join(parts)

`run_react()` uchta guardrailga ega bo'lishi kerak:

- `max_steps` cheksiz siklni cheklaydi;
- `None` yangi mos tool yo'qligini bildiradi;
- planner tarixdagi toolni qayta tanlamaydi.

In [20]:
def run_react(
    query: str, tools: dict, max_steps: int = 3,
    planner=select_tool,
) -> list[dict]:
    """Harakat va kuzatuvlardan iborat cheklangan ReAct sikli."""
    history = []
    for step in range(1, max_steps + 1):
        tool_name = planner(query, history, tools)
        if tool_name is None:
            break
        observation = execute_action(tool_name, query, tools)
        history.append({
            "step": step,
            "tool": tool_name,
            "input": query,
            "observation": observation,
        })
    return history

Endi bir so'rovda ikki xil niyatni yozamiz. Natijada ikki xil tool ishlashi va tarixda ikkita kuzatuv paydo bo'lishi kerak.

In [21]:
query = "Bu kurs yaxshi. RAG haqida ma'lumot ham top."
trace = run_react(query, TOOLS, max_steps=3)

assert [record["tool"] for record in trace] == [
    "retrieve_docs", "sentiment_classify"
]
print(format_final_answer(trace))

retrieve_docs: {"title": "RAG", "text": "RAG avval hujjat topadi, keyin savol va kontekstni generatorga yuboradi.", "score": 1} | sentiment_classify: {"label": "ijobiy", "confidence": 0.82}


Trace agentning kuzatiladigan ish jurnalidir. U yashirin ichki mulohazani emas, bajarilgan harakat va qaytgan kuzatuvni saqlaydi.

In [22]:
for record in trace:
    print(f"{record['step']}-qadam")
    print("  harakat :", record["tool"])
    print("  kuzatuv :", record["observation"])

1-qadam
  harakat : retrieve_docs
  kuzatuv : {'title': 'RAG', 'text': 'RAG avval hujjat topadi, keyin savol va kontekstni generatorga yuboradi.', 'score': 1}
2-qadam
  harakat : sentiment_classify
  kuzatuv : {'label': 'ijobiy', 'confidence': 0.82}


## 7. Guardrail tajribasi

Murakkab so'rov uchta toolga mos tushadi. `max_steps=2` bo'lsa, agent uchinchi harakatga o'tmasligi kerak. Bu limit sifat va xarajat orasidagi boshqariladigan kompromissdir.

In [23]:
complex_query = "Bu yaxshi matn. RAG haqida ma'lumot top va qisqa xulosa qil."
limited_trace = run_react(complex_query, TOOLS, max_steps=2)

assert len(limited_trace) == 2
print("Bajarilgan toollar:", [record["tool"] for record in limited_trace])

Bajarilgan toollar: ['retrieve_docs', 'summarize']


### Qoidaviy plannerning chegarasi

Kalit so'z ishlatilmagan parafrazda planner niyatni tushunmasligi mumkin. Bu tajriba qoidaviy routing va LLM routing orasidagi farqni ko'rsatadi.

In [24]:
paraphrase = "Ushbu yozuvning asosiy fikrlarini ixcham bayon et."
selected = select_tool(paraphrase, [], TOOLS)
print("Tanlangan tool:", selected)
print("Nega bu natija chiqdi? Qaysi tavsif yoki keywordni yaxshilash mumkin?")

Tanlangan tool: None
Nega bu natija chiqdi? Qaysi tavsif yoki keywordni yaxshilash mumkin?


## 8. Yakuniy vazifa — capstone klassi

Endi oldingi funksiyalarni bitta interfeysga yig'amiz:

```text
agent = DocumentAssistantAgent()
answer = agent.run(user_message)
trace = agent.last_trace()
```

Klass tool funksiyalarini qayta implementatsiya qilmaydi. U reyestrni qabul qiladi va bugun yozilgan `run_react()` dan foydalanadi. Keyinchalik `TOOLS` ichidagi demo funksiyalar o'rniga oldingi capstone modullarining metodlarini berish mumkin.

In [25]:
class DocumentAssistantAgent:
    """P15 capstone natijasi: planner va toollari almashtiriladigan agent."""

    def __init__(
        self, tools: Optional[dict] = None, max_steps: int = 3,
        planner=select_tool,
    ):
        self.tools = tools or TOOLS
        self.max_steps = max_steps
        self.planner = planner
        self.history = []

    def run(self, user_message: str) -> str:
        self.history = run_react(
            user_message, self.tools, self.max_steps, self.planner
        )
        return format_final_answer(self.history)

    def last_trace(self) -> list[dict]:
        return list(self.history)

    def reset(self) -> None:
        self.history = []

    def tool_names(self) -> list[str]:
        return list(self.tools)

Shartnoma tekshiruvi capstone klassining tashqi xulqini tekshiradi. Bu katakdan o'tsa, P15 natijasi tayyor hisoblanadi.

In [26]:
agent = DocumentAssistantAgent(max_steps=3)
answer = agent.run("Bu kurs yaxshi. Agent haqida ma'lumot top.")

assert isinstance(answer, str) and answer
assert len(agent.last_trace()) == 2
assert agent.tool_names() == list(TOOLS)
agent.reset()
assert agent.last_trace() == []
print("Capstone shartnomasi: PASS")

Capstone shartnomasi: PASS


## 9. Local LLM — Qwen3 planner

Endi faqat planner qismini lokal LLM bilan almashtiramiz. Controller, toollar va guardrails o'zgarmaydi:

```text
qoidaviy planner ─┐
                  ├─→ bir xil ReAct controller → bir xil toollar
Qwen3 planner ────┘
```

`Qwen/Qwen3-1.7B` Kaggle GPU'da lokal ishlaydi. API kaliti kerak emas, lekin birinchi yuklash uchun **Internet** va GPU kerak. Kaggle paketlari eski bo'lsa, notebook boshida faqat quyidagini o'rnating; `torch`ni qayta o'rnatmang:

```text
%pip install -q -U "transformers>=4.51" accelerate
```

**Dars uchun:** model cache'ga tushishi uchun o'qituvchi bu bo'limni mashg'ulotdan oldin bir marta ishga tushirishi ma'qul.

In [27]:
USE_LOCAL_LLM = True
LOCAL_MODEL_ID = "Qwen/Qwen3-1.7B"

print("Local LLM:", "yoqilgan" if USE_LOCAL_LLM else "o'chirilgan")
print("Model:", LOCAL_MODEL_ID)

Local LLM: yoqilgan
Model: Qwen/Qwen3-1.7B


### 9.1 Modelni bir marta yuklash

Model har bir so'rovda qayta yuklanmaydi. `float16` 1.7B modelni Kaggle T4 xotirasiga joylashtirishga yordam beradi. `device_map="auto"` mavjud GPU'ni tanlaydi.

In [28]:
if USE_LOCAL_LLM:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if not torch.cuda.is_available():
        print("GPU topilmadi — Qwen (local LLM) bo'limi o'tkazib yuboriladi.")
        USE_LOCAL_LLM = False
    else:
        local_tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_ID)
        local_model = AutoModelForCausalLM.from_pretrained(
            LOCAL_MODEL_ID,
            torch_dtype=torch.float16,
            device_map="auto",
        )
        local_model.eval()
        print("GPU:", torch.cuda.get_device_name(0))

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

GPU: Tesla T4


### 9.2 Python reyestridan tool schema yaratish

Qwen tool nomi, tavsifi va argument sxemasini ko'radi. Barcha demo toollar bitta matnli `input` qabul qiladigan umumiy adapter sifatida taqdim etiladi.

In [29]:
def build_tool_schemas(tools: dict) -> list[dict]:
    schemas = []
    for tool_name, tool_info in tools.items():
        schemas.append({
            "type": "function",
            "function": {
                "name": tool_name,
                "description": tool_info["description"],
                "parameters": {
                    "type": "object",
                    "properties": {"input": {"type": "string"}},
                    "required": ["input"],
                },
            },
        })
    return schemas

### 9.3 Strukturali chiqishni tekshirish

Lokal model ham ba'zan formatni buzishi mumkin. Parser javob ichidan birinchi JSON obyektini oladi; noma'lum yoki takroriy toolni guardrail rad etadi.

In [30]:
def first_json_object(text: str) -> dict:
    decoder = json.JSONDecoder()
    for position, character in enumerate(text):
        if character != "{":
            continue
        try:
            value, _ = decoder.raw_decode(text[position:])
            if isinstance(value, dict):
                return value
        except json.JSONDecodeError:
            continue
    raise ValueError(f"Qwen JSON qaytarmadi: {text[:160]}")

def validate_tool_call(text: str, history: list[dict], tools: dict):
    if "FINISH" in text.upper() and "{" not in text:
        return None
    action = first_json_object(text)
    tool_name = action.get("name")
    used_tools = {record["tool"] for record in history}
    if tool_name not in tools or tool_name in used_tools:
        raise ValueError(f"Noto'g'ri yoki takroriy tool: {tool_name}")
    return tool_name

### 9.4 Qwen bilan matn generatsiyasi

`enable_thinking=False` tool tanlashni qisqa va tez qiladi. Modeldan yashirin mulohaza emas, tool-call formatidagi qisqa javob so'raladi.

In [39]:
def generate_qwen_action(messages: list[dict], tools: dict) -> str:
    prompt = local_tokenizer.apply_chat_template(
        messages, tools=build_tool_schemas(tools), tokenize=False,
        add_generation_prompt=True, enable_thinking=False,
    )
    model_inputs = local_tokenizer(prompt, return_tensors="pt").to(local_model.device)
    with torch.inference_mode():
        output_ids = local_model.generate(
            **model_inputs, max_new_tokens=96, do_sample=False
        )
    new_ids = output_ids[0, model_inputs["input_ids"].shape[1]:]
    generated_text = local_tokenizer.decode(new_ids, skip_special_tokens=True)
    print("Qwen chiqishi:", generated_text.strip())
    return generated_text

### 9.5 Qwen planner

Planner so'rov va oldingi kuzatuvlarni modelga beradi. So'ng strukturali chiqishni tekshirib, controller kutadigan tool nomini qaytaradi.

In [32]:
def qwen_select_tool(query: str, history: list[dict], tools: dict):
    history_view = [
        {"tool": item["tool"], "observation": item["observation"]}
        for item in history
    ]
    messages = [
        {"role": "system", "content": (
            "Siz agent plannersiz. Bitta kerakli va ishlatilmagan toolni chaqiring. "
            "Boshqa tool kerak bo'lmasa faqat FINISH yozing."
        )},
        {"role": "user", "content": (
            f"So'rov: {query}\nOldingi kuzatuvlar: "
            f"{json.dumps(history_view, ensure_ascii=False)}"
        )},
    ]
    generated_text = generate_qwen_action(messages, tools)
    try:
        return validate_tool_call(generated_text, history, tools)
    except ValueError as error:
        print("Planner guardrail agentni to'xtatdi:", error)
        return None

### 9.6 Bir xil agent, boshqa planner

Endi capstone klassiga faqat planner funksiyasini beramiz. Qwen toolni tanlaydi; mavjud controller uni bajaradi, kuzatuvni tarixga qo'shadi va keyingi qadamni so'raydi.

In [33]:
if USE_LOCAL_LLM:
    qwen_agent = DocumentAssistantAgent(
        planner=qwen_select_tool,
        max_steps=3,
    )
    qwen_answer = qwen_agent.run(
        "Bu kurs menga manzur bo'ldi. RAG mohiyatini ham tushuntiring."
    )
    print("\nYakuniy javob:", qwen_answer)
    print("Qadamlar:", [item["tool"] for item in qwen_agent.last_trace()])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Qwen chiqishi: <tool_call>
{"name": "summarize", "arguments": {"input": "Bu kurs menga manzur bo'ldi. RAG mohiyatini ham tushuntiring."}}
</tool_call>
Qwen chiqishi: <tool_call>
{"name": "sentiment_classify", "arguments": {"input": "Bu kurs menga manzur bo'ldi. RAG mohiyatini ham tushuntiring."}}
</tool_call>
Qwen chiqishi: <tool_call>
{"name": "summarize", "arguments": {"input": "Bu kurs menga manzur bo'ldi. RAG mohiyatini ham tushuntiring."}}
</tool_call>
Planner guardrail agentni to'xtatdi: Noto'g'ri yoki takroriy tool: summarize

Yakuniy javob: summarize: {"summary": "Bu kurs menga manzur bo'ldi. RAG mohiyatini ham tushuntiring.", "sentence_count": 2} | sentiment_classify: {"label": "neytral", "confidence": 0.55}
Qadamlar: ['summarize', 'sentiment_classify']


**Taqqoslash:** qoidaviy planner faqat yozilgan keywordlarni ko'radi; Qwen esa parafraz ma'nosidan tool tanlashi mumkin. Lekin Qwen sekinroq va ba'zan noto'g'ri format qaytaradi — shu sabab parser, allowlist va `max_steps` saqlanib qoladi.

LangChain ham xuddi shu qismlarni framework sifatida boshqaradi. Bu labda lokal modelning haqiqiy tool-call formati ko'rinishi uchun `transformers` bilan bevosita ishladik.

## 10. Yakun va chiqish savollari

Bugun qurilgan zanjir:

```text
so'rov → planner → tool → observation → history → yakuniy javob
```

Muhokama qiling:

1. Nega `max_steps` agent uchun xavfsizlik chegarasi hisoblanadi?
2. Qoidaviy planner qaysi parafrazlarda xato qiladi?
3. Haqiqiy LLM tool tanlashda `description` va argument sxemasidan qanday foydalanadi?
4. `DocumentAssistantAgent` ga oldingi RAG yoki sentiment capstone modulini qanday ulardingiz?

**Chiqish bileti:** agentning oddiy pipeline'dan eng muhim farqini bir jumlada yozing.

---
## 11. Yakuniy topshiriq qo'shimchalari
Quyidagilar final rubrikasi uchun: `index_kb()`, 5 so'rov demo (3+ vosita),
`capstone/modules/m15_langchain_agent.py` saqlash va FastAPI `POST /predict`.

In [34]:
# 11.1 Konfiguratsiya va bilimlar bazasini yuklash (index_kb)
import os, glob, json

OFFLINE_FALLBACK = os.environ.get("OFFLINE_FALLBACK", "True").lower() == "true"
print("OFFLINE_FALLBACK =", OFFLINE_FALLBACK)

def index_kb(path="uz_kb_mini.jsonl"):
    """uz_kb_mini.jsonl (yoki Kaggle Input) ni KNOWLEDGE_BASE ga yuklaydi."""
    if not os.path.exists(path):
        hits = glob.glob("/kaggle/input/**/uz_kb_mini.jsonl", recursive=True)
        if hits:
            path = hits[0]
    added = 0
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                except json.JSONDecodeError:
                    continue
                text = obj.get("text") or obj.get("content") or ""
                title = obj.get("title") or "hujjat"
                if text:
                    KNOWLEDGE_BASE.append({"title": title, "text": text})
                    added += 1
        print(f"index_kb: {added} ta yozuv yuklandi ({path})")
    else:
        print("uz_kb_mini.jsonl topilmadi — mavjud demo KNOWLEDGE_BASE ishlatiladi.")
    return len(KNOWLEDGE_BASE)

index_kb()

OFFLINE_FALLBACK = True
uz_kb_mini.jsonl topilmadi — mavjud demo KNOWLEDGE_BASE ishlatiladi.


5

In [35]:
# 11.2 Demo: 5 so'rov, kamida 3 xil vosita
agent = DocumentAssistantAgent(max_steps=3, planner=select_tool)

demo_queries = [
    "Agent haqida ma'lumot top",                 # retrieve_docs
    "Bu kurs juda yaxshi bo'ldi",                # sentiment_classify
    "Ushbu matnni qisqa xulosa qil: NLP matnni tahlil qiladi. U tarjima va qidiruvda ishlatiladi.",  # summarize
    "RAG nima? Bu mavzu yaxshi.",                # retrieve + sentiment
    "Transformer haqida ma'lumot top va qisqartir",  # retrieve + summarize
]

all_used = set()
for qx in demo_queries:
    ans = agent.run(qx)
    all_used.update(t["tool"] for t in agent.last_trace())
    print("Q:", qx)
    print("A:", ans, "\n")

print("Ishlatilgan vositalar:", all_used)
assert len(all_used) >= 3, "Kamida 3 xil vosita kerak!"
print("OK: 3+ xil vosita chaqirildi")

Q: Agent haqida ma'lumot top
A: retrieve_docs: {"title": "Agent", "text": "Agent maqsadga qarab vosita tanlaydi, natijani kuzatadi va keyingi qadamni belgilaydi.", "score": 1} 

Q: Bu kurs juda yaxshi bo'ldi
A: sentiment_classify: {"label": "ijobiy", "confidence": 0.82} 

Q: Ushbu matnni qisqa xulosa qil: NLP matnni tahlil qiladi. U tarjima va qidiruvda ishlatiladi.
A: summarize: {"summary": "Ushbu matnni qisqa xulosa qil: NLP matnni tahlil qiladi. U tarjima va qidiruvda ishlatiladi.", "sentence_count": 2} | retrieve_docs: {"title": "NLP", "text": "NLP kompyuterga inson tilidagi matnni tahlil qilishga yordam beradi.", "score": 3} 

Q: RAG nima? Bu mavzu yaxshi.
A: sentiment_classify: {"label": "ijobiy", "confidence": 0.82} | retrieve_docs: {"title": "RAG", "text": "RAG avval hujjat topadi, keyin savol va kontekstni generatorga yuboradi.", "score": 1} 

Q: Transformer haqida ma'lumot top va qisqartir
A: retrieve_docs: {"title": "Transformer", "text": "Transformer attention yordamida tok

In [36]:
# 11.3 Kapstone modulini saqlash: capstone/modules/m15_langchain_agent.py
import os, base64
os.makedirs("capstone/modules", exist_ok=True)
open("capstone/__init__.py", "w").close()
open("capstone/modules/__init__.py", "w").close()

_B64 = "aW1wb3J0IHJlCmltcG9ydCBqc29uCmltcG9ydCBnbG9iCmltcG9ydCBvcwpmcm9tIHR5cGluZyBpbXBvcnQgT3B0aW9uYWwKCktOT1dMRURHRV9CQVNFID0gWwogICAgeyJ0aXRsZSI6ICJOTFAiLCAidGV4dCI6ICJOTFAga29tcHl1dGVyZ2EgaW5zb24gdGlsaWRhZ2kgbWF0bm5pIHRhaGxpbCBxaWxpc2hnYSB5b3JkYW0gYmVyYWRpLiJ9LAogICAgeyJ0aXRsZSI6ICJUcmFuc2Zvcm1lciIsICJ0ZXh0IjogIlRyYW5zZm9ybWVyIGF0dGVudGlvbiB5b3JkYW1pZGEgdG9rZW5sYXIgb3Jhc2lkYWdpIGJvZydsYW5pc2huaSBtb2RlbGxhc2h0aXJhZGkuIn0sCiAgICB7InRpdGxlIjogIlJBRyIsICJ0ZXh0IjogIlJBRyBhdnZhbCBodWpqYXQgdG9wYWRpLCBrZXlpbiBzYXZvbCB2YSBrb250ZWtzdG5pIGdlbmVyYXRvcmdhIHl1Ym9yYWRpLiJ9LAogICAgeyJ0aXRsZSI6ICJBZ2VudCIsICJ0ZXh0IjogIkFnZW50IG1hcXNhZGdhIHFhcmFiIHZvc2l0YSB0YW5sYXlkaSwgbmF0aWphbmkga3V6YXRhZGkgdmEga2V5aW5naSBxYWRhbW5pIGJlbGdpbGF5ZGkuIn0sCiAgICB7InRpdGxlIjogIk8nemJlayBOTFAiLCAidGV4dCI6ICJPJ3piZWsgdGlsaSBhZ2dsdXRpbmF0aXYgYm8nbGdhbmkgdWNodW4gcW8nc2hpbWNoYWxhciBtYXRuIHRhaGxpbGlkYSBtdWhpbS4ifSwKXQoKCmRlZiB0b2tlbml6ZSh0ZXh0OiBzdHIpIC0+IGxpc3Q6CiAgICByZXR1cm4gcmUuZmluZGFsbChyIlthLXpBLVowLTlfyrvKvOKAmSctXSsiLCB0ZXh0Lmxvd2VyKCkpCgoKZGVmIHNlbnRpbWVudF9jbGFzc2lmeSh0ZXh0OiBzdHIpIC0+IGRpY3Q6CiAgICB0b2tlbnMgPSBzZXQodG9rZW5pemUodGV4dCkpCiAgICBwb3NpdGl2ZV93b3JkcyA9IHsieWF4c2hpIiwgImFqb3lpYiIsICJ6bydyIiwgImZveWRhbGkiLCAicXVsYXkifQogICAgbmVnYXRpdmVfd29yZHMgPSB7InlvbW9uIiwgInFpeWluIiwgInNpZmF0c2l6IiwgIm11YW1tbyIsICJzZWtpbiJ9CiAgICBwID0gbGVuKHRva2VucyAmIHBvc2l0aXZlX3dvcmRzKQogICAgbiA9IGxlbih0b2tlbnMgJiBuZWdhdGl2ZV93b3JkcykKICAgIGlmIHAgPiBuOgogICAgICAgIHJldHVybiB7ImxhYmVsIjogImlqb2JpeSIsICJjb25maWRlbmNlIjogMC44Mn0KICAgIGlmIG4gPiBwOgogICAgICAgIHJldHVybiB7ImxhYmVsIjogInNhbGJpeSIsICJjb25maWRlbmNlIjogMC43OX0KICAgIHJldHVybiB7ImxhYmVsIjogIm5leXRyYWwiLCAiY29uZmlkZW5jZSI6IDAuNTV9CgoKZGVmIHJldHJpZXZlX2RvY3MocXVlcnk6IHN0cikgLT4gZGljdDoKICAgIHEgPSBzZXQodG9rZW5pemUocXVlcnkpKQogICAgc2NvcmVkID0gW10KICAgIGZvciBkIGluIEtOT1dMRURHRV9CQVNFOgogICAgICAgIGR0ID0gc2V0KHRva2VuaXplKGRbInRleHQiXSArICIgIiArIGRbInRpdGxlIl0pKQogICAgICAgIHNjb3JlZC5hcHBlbmQoKGxlbihxICYgZHQpLCBkKSkKICAgIGJlc3Rfc2NvcmUsIGJlc3QgPSBtYXgoc2NvcmVkLCBrZXk9bGFtYmRhIHg6IHhbMF0pCiAgICByZXR1cm4geyJ0aXRsZSI6IGJlc3RbInRpdGxlIl0sICJ0ZXh0IjogYmVzdFsidGV4dCJdLCAic2NvcmUiOiBiZXN0X3Njb3JlfQoKCmRlZiBzdW1tYXJpemUodGV4dDogc3RyKSAtPiBkaWN0OgogICAgc2VudHMgPSBbcC5zdHJpcCgpIGZvciBwIGluIHJlLnNwbGl0KHIiWy4hP10rIiwgdGV4dCkgaWYgcC5zdHJpcCgpXQogICAgc2VsID0gc2VudHNbOjJdCiAgICByZXR1cm4geyJzdW1tYXJ5IjogIi4gIi5qb2luKHNlbCkgKyAoIi4iIGlmIHNlbCBlbHNlICIiKSwgInNlbnRlbmNlX2NvdW50IjogbGVuKHNlbCl9CgoKVE9PTFMgPSB7CiAgICAic2VudGltZW50X2NsYXNzaWZ5IjogewogICAgICAgICJmdW5jdGlvbiI6IHNlbnRpbWVudF9jbGFzc2lmeSwKICAgICAgICAiZGVzY3JpcHRpb24iOiAiTWF0biBoaXNzaXlvdGluaSBhbmlxbGF5ZGkuIiwKICAgICAgICAia2V5d29yZHMiOiBbImhpc3NpeW90IiwgImlqb2JpeSIsICJzYWxiaXkiLCAieWF4c2hpIiwgInlvbW9uIiwgImJhaG8iXSwKICAgIH0sCiAgICAicmV0cmlldmVfZG9jcyI6IHsKICAgICAgICAiZnVuY3Rpb24iOiByZXRyaWV2ZV9kb2NzLAogICAgICAgICJkZXNjcmlwdGlvbiI6ICJCaWxpbWxhciBiYXphc2lkYW4gbW9zIGh1amphdG5pIHRvcGFkaS4iLAogICAgICAgICJrZXl3b3JkcyI6IFsidG9wIiwgInFpZGlyIiwgIm1hJ2x1bW90IiwgImh1amphdCIsICJuaW1hIiwgImhhcWlkYSJdLAogICAgfSwKICAgICJzdW1tYXJpemUiOiB7CiAgICAgICAgImZ1bmN0aW9uIjogc3VtbWFyaXplLAogICAgICAgICJkZXNjcmlwdGlvbiI6ICJCZXJpbGdhbiBtYXRubmkgcWlzcWFydGlyYWRpLiIsCiAgICAgICAgImtleXdvcmRzIjogWyJ4dWxvc2EiLCAicWlzcWEiLCAicWlzcWFydGlyIiwgInVtdW1sYXNodGlyIl0sCiAgICB9LAp9CgoKZGVmIHNlbGVjdF90b29sKHF1ZXJ5OiBzdHIsIGhpc3Rvcnk6IGxpc3QsIHRvb2xzOiBkaWN0KSAtPiBPcHRpb25hbFtzdHJdOgogICAgcSA9IHF1ZXJ5Lmxvd2VyKCkKICAgIHVzZWQgPSB7clsidG9vbCJdIGZvciByIGluIGhpc3Rvcnl9CiAgICBiZXN0LCBiZXN0X3MgPSBOb25lLCAwCiAgICBmb3IgbmFtZSwgaW5mbyBpbiB0b29scy5pdGVtcygpOgogICAgICAgIGlmIG5hbWUgaW4gdXNlZDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBzID0gc3VtKDEgZm9yIGt3IGluIGluZm9bImtleXdvcmRzIl0gaWYga3cgaW4gcSkKICAgICAgICBpZiBzID4gYmVzdF9zOgogICAgICAgICAgICBiZXN0LCBiZXN0X3MgPSBuYW1lLCBzCiAgICByZXR1cm4gYmVzdCBpZiBiZXN0X3MgPiAwIGVsc2UgTm9uZQoKCmRlZiBleGVjdXRlX2FjdGlvbih0b29sX25hbWU6IHN0ciwgcXVlcnk6IHN0ciwgdG9vbHM6IGRpY3QpIC0+IGRpY3Q6CiAgICBpZiB0b29sX25hbWUgbm90IGluIHRvb2xzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIk5vbWEnbHVtIHRvb2w6ICIgKyBzdHIodG9vbF9uYW1lKSkKICAgIHJldHVybiB0b29sc1t0b29sX25hbWVdWyJmdW5jdGlvbiJdKHF1ZXJ5KQoKCmRlZiBydW5fcmVhY3QocXVlcnk6IHN0ciwgdG9vbHM6IGRpY3QsIG1heF9zdGVwczogaW50ID0gMywgcGxhbm5lcj1zZWxlY3RfdG9vbCkgLT4gbGlzdDoKICAgIGhpc3RvcnkgPSBbXQogICAgZm9yIHN0ZXAgaW4gcmFuZ2UoMSwgbWF4X3N0ZXBzICsgMSk6CiAgICAgICAgbmFtZSA9IHBsYW5uZXIocXVlcnksIGhpc3RvcnksIHRvb2xzKQogICAgICAgIGlmIG5hbWUgaXMgTm9uZToKICAgICAgICAgICAgYnJlYWsKICAgICAgICBvYnMgPSBleGVjdXRlX2FjdGlvbihuYW1lLCBxdWVyeSwgdG9vbHMpCiAgICAgICAgaGlzdG9yeS5hcHBlbmQoeyJzdGVwIjogc3RlcCwgInRvb2wiOiBuYW1lLCAiaW5wdXQiOiBxdWVyeSwgIm9ic2VydmF0aW9uIjogb2JzfSkKICAgIHJldHVybiBoaXN0b3J5CgoKZGVmIGZvcm1hdF9maW5hbF9hbnN3ZXIoaGlzdG9yeTogbGlzdCkgLT4gc3RyOgogICAgaWYgbm90IGhpc3Rvcnk6CiAgICAgICAgcmV0dXJuICJNb3Mgdm9zaXRhIHRvcGlsbWFkaS4gU28ncm92bmkgYW5pcXJvcSB5b3ppbmcuIgogICAgcmV0dXJuICIgfCAiLmpvaW4oCiAgICAgICAgZiJ7clsndG9vbCddfToge2pzb24uZHVtcHMoclsnb2JzZXJ2YXRpb24nXSwgZW5zdXJlX2FzY2lpPUZhbHNlKX0iIGZvciByIGluIGhpc3RvcnkKICAgICkKCgpjbGFzcyBEb2N1bWVudEFzc2lzdGFudEFnZW50OgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHRvb2xzPU5vbmUsIG1heF9zdGVwczogaW50ID0gMywgcGxhbm5lcj1zZWxlY3RfdG9vbCk6CiAgICAgICAgc2VsZi50b29scyA9IHRvb2xzIG9yIFRPT0xTCiAgICAgICAgc2VsZi5tYXhfc3RlcHMgPSBtYXhfc3RlcHMKICAgICAgICBzZWxmLnBsYW5uZXIgPSBwbGFubmVyCiAgICAgICAgc2VsZi5oaXN0b3J5ID0gW10KCiAgICBkZWYgcnVuKHNlbGYsIHVzZXJfbWVzc2FnZTogc3RyKSAtPiBzdHI6CiAgICAgICAgc2VsZi5oaXN0b3J5ID0gcnVuX3JlYWN0KHVzZXJfbWVzc2FnZSwgc2VsZi50b29scywgc2VsZi5tYXhfc3RlcHMsIHNlbGYucGxhbm5lcikKICAgICAgICByZXR1cm4gZm9ybWF0X2ZpbmFsX2Fuc3dlcihzZWxmLmhpc3RvcnkpCgogICAgZGVmIGxhc3RfdHJhY2Uoc2VsZikgLT4gbGlzdDoKICAgICAgICByZXR1cm4gbGlzdChzZWxmLmhpc3RvcnkpCgogICAgZGVmIHJlc2V0KHNlbGYpOgogICAgICAgIHNlbGYuaGlzdG9yeSA9IFtdCgogICAgZGVmIHRvb2xfbmFtZXMoc2VsZik6CiAgICAgICAgcmV0dXJuIGxpc3Qoc2VsZi50b29scykKCgpkZWYgaW5kZXhfa2IocGF0aDogc3RyID0gInV6X2tiX21pbmkuanNvbmwiKSAtPiBpbnQ6CiAgICBpZiBub3Qgb3MucGF0aC5leGlzdHMocGF0aCk6CiAgICAgICAgaGl0cyA9IGdsb2IuZ2xvYigiL2thZ2dsZS9pbnB1dC8qKi91el9rYl9taW5pLmpzb25sIiwgcmVjdXJzaXZlPVRydWUpCiAgICAgICAgaWYgaGl0czoKICAgICAgICAgICAgcGF0aCA9IGhpdHNbMF0KICAgIGlmIG9zLnBhdGguZXhpc3RzKHBhdGgpOgogICAgICAgIHdpdGggb3BlbihwYXRoLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgICAgICBmb3IgbGluZSBpbiBmOgogICAgICAgICAgICAgICAgbGluZSA9IGxpbmUuc3RyaXAoKQogICAgICAgICAgICAgICAgaWYgbm90IGxpbmU6CiAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBvID0ganNvbi5sb2FkcyhsaW5lKQogICAgICAgICAgICAgICAgZXhjZXB0IGpzb24uSlNPTkRlY29kZUVycm9yOgogICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICB0ID0gby5nZXQoInRleHQiKSBvciBvLmdldCgiY29udGVudCIpIG9yICIiCiAgICAgICAgICAgICAgICBpZiB0OgogICAgICAgICAgICAgICAgICAgIEtOT1dMRURHRV9CQVNFLmFwcGVuZCh7InRpdGxlIjogby5nZXQoInRpdGxlIiwgImh1amphdCIpLCAidGV4dCI6IHR9KQogICAgcmV0dXJuIGxlbihLTk9XTEVER0VfQkFTRSkK"
with open("capstone/modules/m15_langchain_agent.py", "wb") as f:
    f.write(base64.b64decode(_B64))

import importlib, capstone.modules.m15_langchain_agent as m15mod
importlib.reload(m15mod)
from capstone.modules.m15_langchain_agent import DocumentAssistantAgent as _A, index_kb as _ikb
_t = _A()
print("Modul saqlandi va import qilindi. Test:", _t.run("Agent haqida ma'lumot top")[:60])

Modul saqlandi va import qilindi. Test: retrieve_docs: {"title": "Agent", "text": "Agent maqsadga qa


### 11.4 FastAPI — SentimentAPI
`POST /predict` → `{sentiment, confidence}`. Notebook ichida uvicorn fon rejimida.

In [37]:
# Kerakli paketlar (Kaggle Internet yoqilgan bo'lsa)
try:
    import fastapi, uvicorn, nest_asyncio  # noqa
except Exception:
    %pip install -q fastapi uvicorn nest_asyncio requests
    import fastapi, uvicorn, nest_asyncio  # noqa

from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="SentimentAPI")

class Req(BaseModel):
    text: str

@app.post("/predict")
def predict(r: Req):
    result = sentiment_classify(r.text)          # {"label":..,"confidence":..}
    return {"sentiment": result["label"], "confidence": result["confidence"]}

@app.get("/")
def root():
    return {"status": "ok", "endpoint": "POST /predict"}

print("FastAPI app tayyor")

FastAPI app tayyor


In [38]:
# Serverni fon oqimda ishga tushiramiz va /predict ni sinaymiz
import threading, time, uvicorn, nest_asyncio, requests
nest_asyncio.apply()

def _serve():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=_serve, daemon=True).start()
time.sleep(3)

for txt in ["Bu xizmat juda yaxshi", "Sifatsiz va sekin xizmat"]:
    resp = requests.post("http://127.0.0.1:8000/predict", json={"text": txt}, timeout=10)
    data = resp.json()
    print(txt, "->", data)
    assert "sentiment" in data and "confidence" in data, "Format xato!"
print("OK: /predict {sentiment, confidence} qaytardi")

Bu xizmat juda yaxshi -> {'sentiment': 'ijobiy', 'confidence': 0.82}
Sifatsiz va sekin xizmat -> {'sentiment': 'salbiy', 'confidence': 0.79}
OK: /predict {sentiment, confidence} qaytardi


In [41]:
import requests
requests.post("http://127.0.0.1:8000/predict",
              json={"text": "Bu xizmat ajoyib"}).json()

{'sentiment': 'ijobiy', 'confidence': 0.82}